In [2]:
from PIL import Image
from ps3 import PS3VisionModel, PS3ImageProcessor

from transformers import (
    AutoProcessor,
    SiglipImageProcessor,
    SiglipVisionModel)
import torch
from pathlib import Path
from torch import nn


/root/miniconda3/envs/univa/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/root/miniconda3/envs/univa/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [3]:

# Load the PS3 model and processor.
ps3_vision_model = PS3VisionModel.from_pretrained("nvidia/PS3-1.5K-SigLIP2",
    torch_dtype=torch.bfloat16,
                attn_implementation="flash_attention_2",
)
ps3_processor = PS3ImageProcessor.from_pretrained("nvidia/PS3-1.5K-SigLIP2")
ps3_vision_model.cuda().eval()


SIGLIP_PATH="/workspace/UniWorld-V1/model_weight/siglip2-so400m-patch16-512"

siglip_processor = SiglipImageProcessor.from_pretrained(SIGLIP_PATH)
siglip_model = SiglipVisionModel.from_pretrained(
    SIGLIP_PATH,
    torch_dtype=torch.bfloat16,
).to("cuda")

You are attempting to use Flash Attention 2.0 with a model not initialized on GPU. Make sure to move the model to GPU after initializing it on CPU with `model.to('cuda')`.
/root/miniconda3/envs/univa/lib/python3.10/site-packages/ps3/modeling_ps3.py:142: UserWarning: The number of hidden layers to return hidden states is currently set to 27. If this value is large, it can consume a lot of memory. Consider setting it to a smaller value if you won't use all the hidden states from every layer!
  warnings.warn(f"The number of hidden layers to return hidden states is currently set to {self.num_hidden_layers_to_return}. If this value is large, it can consume a lot of memory. Consider setting it to a smaller value if you won't use all the hidden states from every layer!")
Some weights of the model checkpoint at nvidia/PS3-1.5K-SigLIP2 were not used when initializing PS3VisionModel: ['logit_bias', 'logit_scale', 'text_model.ln_final.bias', 'text_model.ln_final.weight', 'text_model.positional_em

In [4]:
SIGLIP_MLP_PATH = Path("/workspace/cleanroom/model_weights/uniworld-stage-2-1024-v2.0.0-explicit-short-prompt/ckpt-6000/siglip_projector.bin")

pretrained_siglip_mlp = torch.load(SIGLIP_MLP_PATH)


/tmp/ipykernel_231838/1930709958.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pretrained_siglip_mlp = torch.load(SIGLIP_MLP_PATH)


In [ ]:

# # You can replace it with your own image.
image = Image.open("/workspace/UniWorld-V1/assets/removal_3.jpg").resize((1024, 1024))


with torch.no_grad():
    x = ps3_processor(image)["pixel_values"][0].unsqueeze(0).cuda().to(torch.bfloat16)
    outs = ps3_vision_model(x, num_look_close=1)
    features = outs.last_hidden_state

print(features.shape)


/root/miniconda3/envs/univa/lib/python3.10/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


AttributeError: 'dict' object has no attribute 'pixel_values'

In [ ]:
x = ps3_processor(images=image, do_resize=True)["pixel_values"][0].unsqueeze(0).cuda().to(torch.bfloat16)


TypeError: BaseImageProcessor.__call__() missing 1 required positional argument: 'images'

In [ ]:

tensors = siglip_processor.preprocess(
        image.convert("RGB"),
        do_resize=True,
        do_convert_rgb=True,
        return_tensors="pt",
    ).pixel_values.cuda()

        
        
siglip_hs = siglip_model(tensors).last_hidden_state

print(siglip_hs.shape) # torch.Size([1, 1024, 1152])


In [ ]:
siglip_projector = nn.Sequential(
    nn.Linear(
        1152,  # HARDCODE, out from siglip
        4096 * 3,  # HARDCODE
    ),
    nn.SiLU(),
    nn.Linear(
        4096 * 3,  # HARDCODE
        4096,  # HARDCODE, context_embedder from flux
    ),
)
# siglip_projector = siglip_projector.to(dtype=torch.bfloat16)

siglip_projector.cuda()

In [ ]:
siglip_hs_projected = siglip_projector(features)

In [ ]:
siglip_hs_projected.shape